# Phần 8: Dự đoán, kiểm tra phiên bản và chạy test

Notebook này dùng model đã lưu để dự đoán giá cho căn nhà **mới** (chưa có
trong dữ liệu), và kiểm tra 3 thứ khiến pipeline dự đoán an toàn để dùng:

1. **Model và môi trường có khớp không** — `model_meta.json` ghi phiên bản
   thư viện lúc lưu; lệch phiên bản có thể làm model đọc sai hoặc lỗi.
2. **Input lạ được xử lý an toàn** — thành phố chưa từng thấy → `Other`;
   thiếu trường → báo lỗi rõ ràng chứ không âm thầm dự đoán sai.
3. **Bộ test tự động** (`tests/`) vẫn xanh.

> Đường dẫn model/schema/meta lấy từ `config.yaml`. Yêu cầu đã chạy
> `preprocessing.py` và `train.py` (hoặc các notebook 02, 03) để có
> `models/best_model.pkl` và `models/feature_schema.json`.

In [ ]:
import sys
sys.path.append('../src')

import json
import pandas as pd

from config import CFG, PROJECT_ROOT, resolve_path
from predict import (load_artifacts, build_features, predict_price, predict_many,
                     validate_raw_input, DEMO_HOUSE)
from utils import library_versions, check_model_compat

## Bước 1: Model đang dùng và phiên bản thư viện

In [ ]:
meta_path = resolve_path(CFG['paths']['model_meta'])
current = library_versions()

if meta_path.exists():
    meta = json.loads(meta_path.read_text())
    print('Model:', meta['model'], '| lưu lúc:', meta['saved_at'], '| seed:', meta['seed'])
    table = pd.DataFrame({
        'lúc lưu model': meta['library_versions'],
        'môi trường hiện tại': current,
    }).fillna('(chưa cài)')
    table['khớp'] = table['lúc lưu model'] == table['môi trường hiện tại']
    display(table) if 'display' in globals() else print(table)

    problems = check_model_compat(meta)
    print('=> Khớp phiên bản.' if not problems else '=> LỆCH: ' + '; '.join(problems))
else:
    print('Chưa có model_meta.json (model lưu trước khi có tính năng này).')
    print('Phiên bản hiện tại:', current)
    print('Chạy lại train.py hoặc tune.py để tạo file này.')

## Bước 2: Dự đoán một căn nhà

`load_artifacts()` tự cảnh báo (`RuntimeWarning`) nếu phiên bản thư viện lệch
so với `model_meta.json`.

In [ ]:
model, schema = load_artifacts()
print('Loại model:', type(model).__name__)
print('Căn nhà mẫu:', json.dumps(DEMO_HOUSE, ensure_ascii=False))

price = predict_price(DEMO_HOUSE, model, schema)
print(f'\n=> Giá dự đoán: ${price:,.0f}')

Vector feature mà model thực sự nhận (chỉ hiện các cột khác 0 để dễ đọc):

In [ ]:
x = build_features(DEMO_HOUSE, schema).iloc[0]
x[x != 0]

## Bước 3: Thành phố chưa từng thấy → `Other`

Lúc train, thành phố ít hơn `min_city_count` mẫu bị gộp vào `Other`.
`predict.py` dùng đúng danh sách thành phố đã lưu trong schema, nên một thành
phố hoàn toàn lạ được xử lý y hệt một thành phố hiếm — không lỗi, không crash.

In [ ]:
unknown = {**DEMO_HOUSE, 'city': 'ThanhPhoLaXaKhongCoTrongData'}
X_unknown = build_features(unknown, schema)

city_cols = [c for c in X_unknown.columns if c.startswith('city_')]
active = [c for c in city_cols if X_unknown[c].iloc[0] == 1]
print('Cột city đang bật:', active)   # kỳ vọng: ['city_Other']

print(f"Giá nhà mẫu ở thành phố lạ: ${predict_price(unknown, model, schema):,.0f}")
print(f"Giá nhà mẫu ở {DEMO_HOUSE['city']}: ${price:,.0f}")

## Bước 4: Thiếu trường → lỗi rõ ràng

Thà báo lỗi còn hơn âm thầm điền giá trị mặc định rồi trả về một con số
trông có vẻ hợp lý nhưng sai.

In [ ]:
incomplete = {k: v for k, v in DEMO_HOUSE.items() if k not in ('sqft_living', 'yr_built')}
try:
    validate_raw_input(incomplete)
except ValueError as e:
    print('Lỗi (đúng như kỳ vọng):', e)

## Bước 5: Dự đoán nhiều căn nhà từ file JSON

In [ ]:
houses = json.loads((PROJECT_ROOT / 'examples' / 'sample_houses.json').read_text())
prices = predict_many(houses)

pd.DataFrame({
    'city': [h['city'] for h in houses],
    'sqft_living': [h['sqft_living'] for h in houses],
    'giá dự đoán (USD)': [round(p) for p in prices],
})

## Bước 6: Chạy bộ test

Bộ test dùng dữ liệu giả sinh trong `tests/conftest.py` nên không phụ thuộc
vào dữ liệu hay model của bạn. Cần cài `pytest`
(`pip install -r requirements-dev.txt`); nếu chưa cài, cell dưới sẽ báo
`No module named pytest`.

Test quan trọng nhất là `test_train_serve_consistency`: nó đảm bảo một căn nhà
qua pipeline lúc train và qua `predict.py` cho ra **đúng cùng vector
feature** — kiểm tra tự động cho điều đã kiểm tra bằng mắt ở notebook 02.

In [ ]:
import subprocess

result = subprocess.run(
    [sys.executable, '-m', 'pytest', '-q'],
    cwd=PROJECT_ROOT, capture_output=True, text=True,
)
print(result.stdout[-3000:])
if result.returncode != 0:
    print(result.stderr[-1500:])
print('Mã thoát:', result.returncode, '(0 = tất cả test qua)')